In [22]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
from collections import Counter

# Konfigurasi path
DATA_PATH = '../../datapreparation/datapreprocessing-no-stem/data_preprocessing_final_no_stem.csv'
SLA_LEXICON_PATH = '../outputs/sla_lexicon_adapted.csv'
OUTPUT_DIR = '../outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [23]:
# 2.2 Load Data Ground Truth
df = pd.read_csv(DATA_PATH)

print(f"\n[DATA] Ground truth berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")

df.head()


[DATA] Ground truth berhasil dimuat: 13192 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...


In [24]:
# 2.3 Load Leksikon SLA yang Telah Diadaptasi
df_sla = pd.read_csv(
    SLA_LEXICON_PATH,
    sep='\t',                  
    header=None,               
    names=['kata', 'mean', 'std', 'raw_ratings']
)

df_sla['kata'] = df_sla['kata'].astype(str).str.strip().str.lower()

# Dictionary lexicon untuk matching
sla_dict = dict(zip(df_sla['kata'], df_sla['mean']))

print(f"[LEKSIKON] SLA berhasil dimuat: {len(sla_dict)} entri.")

[LEKSIKON] SLA berhasil dimuat: 9855 entri.


In [25]:
# 2.4 Definisi Kategori Kata Fungsi (Untuk Ignore Function)
# Kategori ini akan digunakan untuk netralisasi (skor=0) di tahap scoring nanti
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}

KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}

PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}

PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

# Gabungan semua kata fungsi
ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [26]:
# 2.5 Konfigurasi Ignore Set
# Hanya kata fungsi yang ADA di leksikon yang akan dinetralkan
ignore_set_final = set(df_sla[df_sla['kata'].isin(ALL_FUNCTION_WORDS)]['kata'])

print(f"\n[KONFIGURASI IGNORE FUNCTION]")
print(f"Total kata fungsi dalam definisi: {len(ALL_FUNCTION_WORDS)} kata")
print(f"Kata fungsi yang ada di leksikon (akan dinetralkan): {len(ignore_set_final)} kata")
print(f"\nDaftar kata fungsi yang akan dinetralkan (skor=0):")
print(f"  {sorted(ignore_set_final)}")


[KONFIGURASI IGNORE FUNCTION]
Total kata fungsi dalam definisi: 44 kata
Kata fungsi yang ada di leksikon (akan dinetralkan): 22 kata

Daftar kata fungsi yang akan dinetralkan (skor=0):
  ['aku', 'anda', 'apa', 'boleh', 'bukan', 'dari', 'dia', 'harus', 'itu', 'jangan', 'karena', 'oleh', 'pada', 'pun', 'sangat', 'saya', 'siapa', 'sudah', 'tidak', 'wajib', 'ya', 'yang']


In [27]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    """Tokenisasi sederhana berdasarkan spasi"""
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

total_tokens = df['tokens'].str.len().sum()
print(f"\n[TOKENISASI] Selesai. Total token: {total_tokens:,}")


[TOKENISASI] Selesai. Total token: 235,559


In [28]:
# 2.7 Fungsi Lexicon Matching dengan Ignore Function
def match_lexicon_with_ignore(tokens, lexicon, ignore_set):
    """
    Memisahkan token menjadi 3 kategori:
    1. Matched: Kata yang ada di leksikon (bukan fungsi)
    2. Ignored: Kata fungsi yang akan dinetralkan
    3. Unmatched: Kata yang tidak ada di leksikon
    """
    matched_words = []
    ignored_words = []
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in ignore_set:
            ignored_words.append(token)
        elif token_lower in lexicon:
            matched_words.append(token)
        else:
            unmatched_words.append(token)
            
    return matched_words, ignored_words, unmatched_words

In [29]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching SLA dengan ignore function...")

df[['matched_words', 'ignored_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(
        lambda x: match_lexicon_with_ignore(x, sla_dict, ignore_set_final)
    ).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching SLA dengan ignore function...
[INFO] Lexicon matching selesai.


In [30]:
# 2.9 Perhitungan Statistik Matching
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_ignored = df['ignored_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

print("[STATISTIK] Hasil Lexicon Matching (SLA + Ignore Function):")
print(f"Total kata             : {total_words:,}")
print(f"Matched di SLA         : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"Ignored (fungsi)       : {total_ignored:,} ({(total_ignored/total_words)*100:.2f}%)")
print(f"Unmatched              : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print("-"*60)
print(f"Coverage Rate          : {((total_matched + total_ignored)/total_words)*100:.2f}%")
print(f"  (Matched + Ignored)")

[STATISTIK] Hasil Lexicon Matching (SLA + Ignore Function):
Total kata             : 235,559
Matched di SLA         : 141,410 (60.03%)
Ignored (fungsi)       : 12,855 (5.46%)
Unmatched              : 81,294 (34.51%)
------------------------------------------------------------
Coverage Rate          : 65.49%
  (Matched + Ignored)


In [31]:
# 2.10 Analisis Top Unmatched Words
print("\n[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:")
print("-"*40)

all_unmatched = []
for lst in df['unmatched_words']:  
    all_unmatched.extend([w.lower() for w in lst])

unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

print(f"{'Kata':<25} | {'Frekuensi':<10}")
print("-"*40)
for word, freq in top_50_unmatched:
    print(f"{word:<25} | {freq:<10}")


[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:
----------------------------------------
Kata                      | Frekuensi 
----------------------------------------
di                        | 3158      
dan                       | 3099      
ini                       | 1870      
?                         | 1613      
untuk                     | 1508      
dengan                    | 1430      
ke                        | 1365      
!                         | 932       
jika                      | 764       
akan                      | 700       
menjadi                   | 548       
bisa                      | 543       
tetapi                    | 486       
juga                      | 454       
kali                      | 427       
saat                      | 400       
tak                       | 383       
terhadap                  | 291       
telah                     | 284       
baru                      | 283       
nya                       | 281       
atau

In [32]:
# 2.11 Preview Hasil Matching
print("[PREVIEW] 5 Tweet Pertama:")

for i in range(min(5, len(df))):
    print(f"\n[Tweet {i+1}]")
    teks_preview = df['teks_processed'].iloc[i]
    if len(teks_preview) > 80:
        teks_preview = teks_preview[:80] + "..."
    print(f"Teks: {teks_preview}")
    print(f"  ✓ Matched : {df['matched_words'].iloc[i]}")
    print(f"  ⊘ Ignored : {df['ignored_words'].iloc[i]}")
    print(f"  ✗ Unmatched: {df['unmatched_words'].iloc[i][:5]}...")  

[PREVIEW] 5 Tweet Pertama:

[Tweet 1]
Teks: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !
  ✓ Matched : ['ADIL', 'punya', 'kebijakan', 'publik', 'negara', 'ingat']
  ⊘ Ignored : ['yang', 'yang']
  ✗ Unmatched: ['loh', 'untuk', 'ini', '!', '!']...

[Tweet 2]
Teks: tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kep...
  ✓ Matched : ['media', 'DPR', 'pemerintah', 'sporadis', 'selektif', 'hanya', 'kepada', 'media', 'berseberangan']
  ⊘ Ignored : ['jangan', 'yang']
  ✗ Unmatched: ['tertibkan', 'online', 'apalagi']...

[Tweet 3]
Teks: harus dievaluasi lagi kebijakan bebas visa terutama untuk negara tiongkok pak ! ...
  ✓ Matched : ['lagi', 'kebijakan', 'bebas', 'visa', 'terutama', 'negara', 'pak', 'bahaya', 'martabat', 'RI']
  ⊘ Ignored : ['harus']
  ✗ Unmatched: ['dievaluasi', 'untuk', 'tiongkok', '!', '!']...

[Tweet 4]
Teks: jangan ngambang aturan logis apa undang undang
  ✓ Matched : ['aturan', 'undang', 'undang']
  ⊘ Ignored : ['janga

In [33]:
# 2.12 Simpan Output Matching
output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching.csv')
df.to_csv(output_path, index=False)

print("[OUTPUT] Data berhasil disimpan!")
print(f"File: {output_path}")

print("\n[CATATAN PENTING]")
print("Tahap ini HANYA untuk matching dan statistik.")
print("Netralisasi kata fungsi (skor=0) akan diterapkan di:")
print("Notebook 3: Vader Scoring")

[OUTPUT] Data berhasil disimpan!
File: ../outputs\lexicon_matching.csv

[CATATAN PENTING]
Tahap ini HANYA untuk matching dan statistik.
Netralisasi kata fungsi (skor=0) akan diterapkan di:
Notebook 3: Vader Scoring
